In [187]:
import numpy as np
import pdb
import pandas as pd
import Bio
from Bio import SeqIO
import urllib.request


In [17]:
def gc_load(i):
    n = len(i)
    m = 0
    
    for c in i:
        if c == 'G' or c == 'C':
            m += 1
    
    return 100*(float(m)/n)

In [29]:
def result_gc():
    max_gc = 0
    for record in SeqIO.parse('datasets/rosalind_gc.txt','fasta'):
        if gc_load(record.seq) > max_gc:
            max_gc = gc_load(record.seq)
            max_gc_record_name = record.name
    print (max_gc_record_name, max_gc)

In [58]:
result_gc()

Rosalind_2767 51.216814159292035


## End of GC

## Start of Cons

Starting Consensus and Profile, http://rosalind.info/problems/cons/

Plan = 1. Read in strings, 2. Build profile Matrix, 3. Generate consensus string, 4. Handle output.

In [123]:
def profile_matrix():
    dataset = SeqIO.parse('datasets/rosalind_cons.txt','fasta')
    
    string_count = 0
    string_length = 0
    
    for index, record in enumerate(dataset):
        string_length = len(record.seq)
        string_count += 1
    
#     print(string_count, string_length)
    
    df = pd.DataFrame(0, index={'A:', 'C:', 'G:', 'T:'}, columns=range(0, string_length)).sort_index(axis=0)
    #print(df)
    
    dataset = SeqIO.parse('datasets/rosalind_cons.txt','fasta')
    
    for index, record in enumerate(dataset):
        #print(index,record)
        for index, letter in enumerate(record.seq):
            #print(index, letter)
            if letter == 'A':
                df.at['A:',index] += 1
            if letter == 'C':
                df.at['C:',index] += 1
            if letter == 'G':
                df.at['G:',index] += 1
            if letter == 'T':
                df.at['T:',index] += 1
    
    return(df)

In [107]:
def consensus(matrix):
    return(matrix.idxmax(axis=0))

In [131]:
def solve_cons():
    with open('results/results_cons.txt', 'w+') as f:
        for value in (consensus(profile_matrix())):
            f.write(value.replace(':',''))
        f.write('\n')
        f.write(profile_matrix().to_string(header=False))
        
solve_cons()

## End of cons, Beginning of prtm

A   71.03711
C   103.00919
D   115.02694
E   129.04259
F   147.06841
G   57.02146
H   137.05891
I   113.08406
K   128.09496
L   113.08406
M   131.04049
N   114.04293
P   97.05276
Q   128.05858
R   156.10111
S   87.03203
T   101.04768
V   99.06841
W   186.07931
Y   163.06333 

In [160]:
def solve_prtm():
    protein_weights = {'A': 71.03711, 'C': 103.00919, 'D': 115.02694, 'E': 129.04259, 'F': 147.06841, 'G': 57.02146, 'H': 137.05891, 'I': 113.08406, 'K': 128.09496, 'L': 113.08406, 'M': 131.04049, 'N': 114.04293, 'P': 97.05276, 'Q': 128.05858, 'R': 156.10111, 'S': 87.03203, 'T': 101.04768, 'V': 99.06841, 'W': 186.07931, 'Y': 163.06333} 
    with open('datasets/rosalind_prtm.txt') as f:
        read_data = f.read()
    f.closed
    total_weight = 0
    for char in list(read_data):
        total_weight += (protein_weights[char])
    return total_weight

solve_prtm()

111319.01047000055

## End of prtm -- Start of mprt

mprt plan : 
#. get records for the list of proteins into a dataframe
#. crawl through the sequences to find the pattern : N, not P, S or T, Not P

In [268]:
import re
import sys
import io
from Bio import SeqIO
import requests
import urllib


if __name__ == '__main__':
    read_data = open('datasets/rosalind_mprt.txt').readlines()
    for value in read_data:
        value = value.strip()
        address = ('http://www.uniprot.org/uniprot/' + value + '.fasta')
        print(address)
        with urllib.request.urlopen(address) as response:
                payload = response.read().strip()
#                 print(payload)

        s = SeqIO.read(payload, 'fasta')
        print(s)
#         mos = [x for x in re.finditer(r'(?=(N[^P][ST][^P]))',  str(s.seq))]
#         if not len(mos):
#             continue
#         print(acc)
#         print(' '.join([str(mo.start(0) + 1) for mo in mos]))

http://www.uniprot.org/uniprot/P.fasta


HTTPError: HTTP Error 404: Not Found

In [270]:
import re
import sys
import io
from Bio import SeqIO
import requests


if __name__ == '__main__':
    accs = open('datasets/rosalind_mprt.txt').read().strip().split('\n')
    for acc in accs:
        r = requests.get('http://www.uniprot.org/uniprot/%s.fasta' % acc)
        s = SeqIO.read(io.StringIO(r.text), 'fasta')
        mos = [x for x in re.finditer(r'(?=(N[^P][ST][^P]))',  str(s.seq))]
        if not len(mos):
            continue
        print(acc)
        print(' '.join([str(mo.start(0) + 1) for mo in mos]))

P04180_LCAT_HUMAN
44 108 296 408
Q2V4D9
64
P07357_CO8A_HUMAN
43 437
P10761_ZP3_MOUSE
146 273 304 327 330
Q5U1Y9
102
P13838_LEUK_RAT
274 300
P02748_CO9_HUMAN
277 415
Q7TMB8
209 291 328 442 607 672 831 858
Q90304_C166_CARAU
92 171 350 441 465
P01233_CGHB_HUMAN
33 50
P00749_UROK_HUMAN
322
P39873_RNBR_BOVIN
88
